# Task 14 — SpatialVacuum Lean diagnostic

Fresh public clone at an immutable raw SHA. CPU/high RAM, exact Lean/mathlib pins, focused target only.

In [ ]:
import datetime, hashlib, os, platform, subprocess, tempfile, time
from pathlib import Path

EXPECTED_SHA = 'd685e82e10c8cd64221444a03ff83f617cec96b4'
EXPECTED_TOOLCHAIN = 'leanprover/lean4:v4.29.0-rc6'
EXPECTED_LEAN_COMMIT = '00659f8e6071d7e46131ed643bf8003b99b044e9'
EXPECTED_MATHLIB = '07642720480157414db592fa85b626dafb71355b'
REPO_URL = 'https://github.com/lluiseriksson/THE-ERIKSSON-PROGRAMME.git'
WORK = Path(tempfile.mkdtemp(prefix='spatial-vacuum-lean-'))
REPO = WORK / 'repo'
transcript = []

def log(message):
    text = str(message)
    transcript.append(text)
    print(text, flush=True)

def run(cmd, *, cwd=None, env=None, check=True):
    shown = ' '.join(map(str, cmd))
    log(f'$ {shown}')
    start = time.perf_counter()
    process = subprocess.run(cmd, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    elapsed = time.perf_counter() - start
    log(process.stdout.rstrip())
    log(f'[exit {process.returncode}; elapsed {elapsed:.6f} s]')
    if check and process.returncode:
        raise RuntimeError(f'command failed: {shown}')
    return process, elapsed

utc_start = datetime.datetime.now(datetime.timezone.utc)
log(f'utc_start={utc_start.isoformat()}')
log(f'runtime={platform.platform()}')
log(f'python={platform.python_version()}')
log(f'cpu_count={os.cpu_count()}')
log(Path('/proc/cpuinfo').read_text(errors='replace').split('model name', 1)[1].splitlines()[0].lstrip('\t: '))
log(Path('/proc/meminfo').read_text(errors='replace').splitlines()[0])
log('gpu=none (CPU runtime requested)')
run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO)])
run(['git', 'checkout', '--detach', EXPECTED_SHA], cwd=REPO)
head = run(['git', 'rev-parse', 'HEAD'], cwd=REPO)[0].stdout.strip()
if head != EXPECTED_SHA:
    raise RuntimeError(f'HEAD mismatch: {head}')
toolchain = (REPO / 'lean-toolchain').read_text().strip()
manifest = (REPO / 'lake-manifest.json').read_text()
if toolchain != EXPECTED_TOOLCHAIN or EXPECTED_MATHLIB not in manifest:
    raise RuntimeError('toolchain or mathlib pin mismatch')
installer = WORK / 'elan-init.sh'
run(['curl', '--proto', '=https', '--tlsv1.2', '-sSfL', 'https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh', '-o', str(installer)])
installer_sha = hashlib.sha256(installer.read_bytes()).hexdigest()
log(f'elan_installer_sha256={installer_sha}')
run(['sh', str(installer), '-y', '--no-modify-path', '--default-toolchain', 'none'])
env = os.environ.copy()
env['PATH'] = str(Path.home() / '.elan' / 'bin') + os.pathsep + env['PATH']
env['ELAN_TOOLCHAIN'] = EXPECTED_TOOLCHAIN
run(['elan', 'toolchain', 'install', EXPECTED_TOOLCHAIN], env=env)
lean_version = run(['lean', '--version'], env=env)[0].stdout
if EXPECTED_LEAN_COMMIT not in lean_version:
    raise RuntimeError('Lean commit mismatch')
run(['lake', 'exe', 'cache', 'get'], cwd=REPO, env=env)
build, build_seconds = run(['lake', 'build', 'YangMills.OS.SpatialVacuum'], cwd=REPO, env=env, check=False)
log(f'utc_end={datetime.datetime.now(datetime.timezone.utc).isoformat()}')
log(f'build_seconds={build_seconds:.6f}')
log(f'focused_exit={build.returncode}')
Path('/content/spatial_vacuum_lean_transcript.txt').write_text('\n'.join(transcript) + '\n', encoding='utf-8')
transcript_hash = hashlib.sha256(Path('/content/spatial_vacuum_lean_transcript.txt').read_bytes()).hexdigest()
print(f'transcript_sha256={transcript_hash}', flush=True)
if build.returncode:
    raise RuntimeError('SPATIAL VACUUM LEAN FAIL')
print('SPATIAL VACUUM LEAN FOCUSED PASS', flush=True)
from google.colab import files
files.download('/content/spatial_vacuum_lean_transcript.txt')
